# Model: OA Damper Stuck (Experimental Dataset, Cross-Season Evaluation)

## Design: cross-season generalization, not TimeSeriesSplit

Per build_experimental_features.py's design notes: each file is only ~1-2 days, so
a within-file time split isn't meaningful. The real, honest evaluation here is
cross-season generalization - train on some seasons' baseline+fault data, evaluate
on a held-out season the model has never seen. This is the season-dataset analog of
the Simulated dataset's forward-in-time TimeSeriesSplit evaluation, and directly
tests what notebooks 08-10 already probed manually (does a finding from one season
hold in another) - just formalized as a real train/test split with a real model.

## Feature choice, informed by notebook 08's EDA

Per notebook 08: RTU_OA_DMPR_DM detects this fault trivially and directly.
RTU_SA_TEMP is fully compensated/masked - deliberately EXCLUDED here, since
including a feature proven blind to the fault would only add noise. Using
RTU_OA_DMPR_DM and RTU_OA_TEMP together (not MA_TEMP-OA_TEMP directly, since that
was a manual EDA derivation - letting the model use both raw signals and see if it
extracts an equivalent relationship on its own is a fair, honest test).

## Seasons to use

Per notebook 08: Fall_2020 was flagged as structurally atypical (notebook 07 found
three independent oddities). Using Winter_2022 and Spring_2021 for train (the two
seasons where OA damper stuck was actually validated in the EDA), holding out
Summer_2021 as the cross-season test (never examined for this fault in the EDA,
a genuinely fresh test) - Fall_2020 excluded entirely given its flagged oddities,
consistent with treating it as a special case rather than a normal season.

In [1]:
import sys
from pathlib import Path

import pandas as pd

ml_root = Path.cwd().parent
if str(ml_root) not in sys.path:
    sys.path.insert(0, str(ml_root))

from sklearn.ensemble import RandomForestClassifier  # noqa: E402
from sklearn.metrics import classification_report  # noqa: E402
from src.features.build_experimental_features import build_experimental_feature_table  # noqa: E402

feature_cols = ("RTU_OA_DMPR_DM", "RTU_OA_TEMP")

train_table = pd.concat([
    build_experimental_feature_table(
        baseline_path=f"../data/raw/experimental/ERTU_{season}.csv",
        fault_paths={
            f"damper_005_{season}": f"../data/raw/experimental/OA_damper_stuck_005_{season}.csv",
            f"damper_010_{season}": f"../data/raw/experimental/OA_damper_stuck_010_{season}.csv",
            f"damper_050_{season}": f"../data/raw/experimental/OA_damper_stuck_050_{season}.csv",
            f"damper_100_{season}": f"../data/raw/experimental/OA_damper_stuck_100_{season}.csv",
        },
        feature_cols=feature_cols,
    )
    for season in ["Winter_2022", "Spring_2021"]
], ignore_index=True)

test_table = build_experimental_feature_table(
    baseline_path="../data/raw/experimental/ERTU_Summer_2021.csv",
    fault_paths={
        "damper_005_Summer_2021": "../data/raw/experimental/OA_damper_stuck_005_Summer_2021.csv",
        "damper_010_Summer_2021": "../data/raw/experimental/OA_damper_stuck_010_Summer_2021.csv",
        "damper_050_Summer_2021": "../data/raw/experimental/OA_damper_stuck_050_Summer_2021.csv",
        "damper_100_Summer_2021": "../data/raw/experimental/OA_damper_stuck_100_Summer_2021.csv",
    },
    feature_cols=feature_cols,
)

print(f"Train (Winter+Spring) shape: {train_table.shape}, labels:\n{train_table['label'].value_counts()}")
print(f"\nTest (Summer, held out) shape: {test_table.shape}, labels:\n{test_table['label'].value_counts()}")

Train (Winter+Spring) shape: (10800, 5), labels:
label
1    7200
0    3600
Name: count, dtype: int64

Test (Summer, held out) shape: (5400, 5), labels:
label
1    3600
0    1800
Name: count, dtype: int64


## Cross-season evaluation: train on Winter+Spring, test on held-out Summer

In [2]:
X_train = train_table[list(feature_cols)]
y_train = train_table["label"]
X_test = test_table[list(feature_cols)]
y_test = test_table["label"]

model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("=== Cross-season test: trained on Winter+Spring, evaluated on Summer ===")
print(classification_report(y_test, y_pred, target_names=["baseline", "damper_stuck"]))


=== Cross-season test: trained on Winter+Spring, evaluated on Summer ===
              precision    recall  f1-score   support

    baseline       1.00      0.11      0.20      1800
damper_stuck       0.69      1.00      0.82      3600

    accuracy                           0.70      5400
   macro avg       0.85      0.56      0.51      5400
weighted avg       0.80      0.70      0.61      5400



## Cross-season result: baseline recall collapses (0.11) on Summer — a direct,
## expected confirmation of notebook 07's season-confound finding, not a new problem

Baseline precision is perfect (1.00 - when the model says baseline, it's always
right) but recall is only 0.11 - it almost never recognizes Summer's genuine
baseline as normal. This is not a mysterious failure; it directly matches notebook
07's own finding: Summer's baseline economizer position sits near its ~6-7% minimum
(since Summer's outdoor temperature rarely drops below the documented 50°F enable
threshold), while the model was trained on Winter/Spring, where baseline genuinely
shows the economizer active (~22% open). To a model trained on "normal = ~22% open,"
Summer's genuinely normal ~6-7% looks like a damper stuck near minimum - which is
LITERALLY what several of the fault severities (5%, 10%) actually are.

**This is a real, honest, and valuable finding**: it confirms that season-specific
behavior is not just a nuisance for EDA comparisons - it's a real, measurable risk
for any cross-season deployment of this exact feature set. A model trained on
cooler-season data cannot be trusted to correctly recognize warm-season normal
operation using RTU_OA_DMPR_DM/RTU_OA_TEMP alone, since warm-season normal and
mild-fault conditions both look like "damper near minimum."

**Practical implication, a real fix worth testing**: this suggests the model needs
season as an explicit input feature (or season-specific models), rather than being
trained to generalize across seasons using only OA_TEMP and damper position. This
mirrors the "MA_TEMP - OA_TEMP" normalization insight from the EDA itself - the
raw damper position alone conflates "normal for this season" with "abnormal
regardless of season," and the fix is likely a relative/contextual feature, not
raw values, same lesson as the Simulated dataset's weather-residualization
approach, just requiring a season-aware version here.

## Correcting course: literal season-as-category can't generalize to a held-out
## season; using OA_TEMP-based residualization instead, per the Simulated-dataset
## approach

Adding season as a categorical feature would fail structurally here - the model
would have never seen "Summer" as a category during training, making it useless at
test time (this is fundamentally different from the Simulated dataset's continuous
weather variable, which the model CAN interpolate/extrapolate along).

The real, functional relationship is RTU_OA_DMPR_DM as a function of RTU_OA_TEMP
(the documented economizer control logic), not season directly - season is only a
loose proxy for typical OA_TEMP. Fitting a baseline-only regression of damper
position on OA_TEMP using TRAINING data (Winter+Spring), then using the RESIDUAL
(actual damper position minus what that regression predicts for that OA_TEMP) as
the feature, should let the model recognize "expected near-minimum position, given
today's OA_TEMP" as normal - even in Summer, which it's never seen, because the
learned relationship is a continuous function of temperature, not a memorized
per-season average.

In [3]:
from sklearn.linear_model import LinearRegression

train_baseline_rows = train_table[train_table["label"] == 0]
damper_weather_model = LinearRegression()
damper_weather_model.fit(train_baseline_rows[["RTU_OA_TEMP"]], train_baseline_rows["RTU_OA_DMPR_DM"])

train_table["damper_residual"] = train_table["RTU_OA_DMPR_DM"] - damper_weather_model.predict(train_table[["RTU_OA_TEMP"]])
test_table["damper_residual"] = test_table["RTU_OA_DMPR_DM"] - damper_weather_model.predict(test_table[["RTU_OA_TEMP"]])

print("Residual summary by label, TRAIN:")
print(train_table.groupby("label")["damper_residual"].describe())
print("\nResidual summary by label, TEST (Summer, held out):")
print(test_table.groupby("label")["damper_residual"].describe())

Residual summary by label, TRAIN:
        count          mean        std        min        25%       50%  \
label                                                                    
0      3600.0  2.400055e-15  23.235625 -77.125448 -17.860557 -7.575626   
1      7200.0  1.162402e+01  53.195193 -74.899023 -38.727198  0.144802   

             75%         max  
label                         
0      16.497741   67.406910  
1      57.953467  124.934337  

Residual summary by label, TEST (Summer, held out):
        count       mean        std        min        25%        50%  \
label                                                                  
0      1800.0  25.880851  13.403072 -17.134776  15.495071  29.552051   
1      3600.0  59.968061  40.335360 -10.350378  28.111149  50.388099   

             75%         max  
label                         
0      37.067856   45.989844  
1      84.132232  134.632151  


## Partial improvement, not a full fix — testing the classifier directly

Summer's baseline residual mean (25.88) is not centered at 0 as hoped - the
Winter/Spring-fit OA_TEMP-vs-damper relationship doesn't perfectly transfer to
Summer, echoing the undercharge/weather-residualization pattern (helps, doesn't
fully resolve). But there IS real separation between baseline (mean 25.88) and
fault (mean 59.97) residuals in the held-out test set - checking whether this
translates into better classifier performance than the raw-feature version.

In [4]:
model_resid = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
model_resid.fit(train_table[["damper_residual"]], train_table["label"])
y_pred_resid = model_resid.predict(test_table[["damper_residual"]])

print("=== Cross-season test WITH damper_residual feature ===")
print(classification_report(y_test, y_pred_resid, target_names=["baseline", "damper_stuck"]))

=== Cross-season test WITH damper_residual feature ===
              precision    recall  f1-score   support

    baseline       0.42      0.34      0.38      1800
damper_stuck       0.70      0.77      0.73      3600

    accuracy                           0.63      5400
   macro avg       0.56      0.55      0.56      5400
weighted avg       0.61      0.63      0.61      5400



## Result: OA_TEMP-residualization gives a real, partial improvement — not a
## full fix, a genuine precision/recall tradeoff

| | Raw features | With damper_residual |
|---|---|---|
| Baseline recall | 0.11 | 0.34 |
| Baseline precision | 1.00 | 0.42 |

Residualizing damper position against OA_TEMP triples baseline recall (better at
recognizing Summer's genuine normal operation) but at a real cost to precision
(more false alarms - the model now sometimes flags real baseline as faulty too).
This is a genuine, incomplete fix - consistent with the broader pattern across this
whole project that weather/season-driven confounds are real, partially addressable,
but rarely fully solved by a single residualization step.

**Honest conclusion for this fault, cross-season deployment**: neither the raw
feature set nor the residualized version fully solves cross-season generalization
for OA damper stuck. The raw version is falsely confident (perfect precision, poor
recall - misses real baseline constantly). The residualized version trades some of
that false confidence for real detection improvement, at the cost of more false
alarms. Which tradeoff is preferable is a genuine product decision (same category
as the Isolation Forest's contamination tradeoff), not something to resolve
unilaterally here.

**Practical recommendation, until further work is done**: do not deploy a single
cross-season model for this fault without either (a) season-specific baselines/
thresholds (the most direct fix, not yet built), or (b) accepting one of these two
tradeoffs explicitly and monitoring accordingly. This is a real, flagged limitation
of the current pipeline, not a solved problem.

## Summary: OA damper stuck, cross-season model (Experimental dataset)

First model built on the Experimental dataset, using a new build_experimental_
feature_table() module and a new evaluation design (cross-season generalization,
replacing TimeSeriesSplit since individual files are too short for a meaningful
within-file time split).

**Real finding**: a model trained on Winter+Spring, tested on held-out Summer,
shows a severe cross-season generalization problem - directly explained by, and
consistent with, notebook 07's own finding that baseline damper position varies
substantially by season (~22% active in Winter/Spring vs. ~6-7% near-minimum in
Summer). Raw features: perfect precision, poor recall (0.11). OA_TEMP-residualized
features: real improvement to recall (0.34) at a real cost to precision (0.42) -
a genuine, unresolved tradeoff, not a clean fix.

**Status**: first Experimental-dataset model built and honestly evaluated. Confirms
this dataset's cross-season risk is real and consequential for deployment, not just
an EDA curiosity - a genuinely valuable, if sobering, finding to close the day on.